In [7]:
import pandas as pd
import os
import glob
from pathlib import Path

In [ ]:
def cargar_csv():
    """
    Carga los archivos principales de la EPA y los anexos
    desde data/raw y devuelve dos diccionarios:
    - dfs: archivos principales
    - dfs_a: anexos
    """

    ruta = Path("../data/raw")  # carpeta donde están los datos

    dfs = {}    # aquí guardamos los df principales
    dfs_a = {}  # aquí guardamos los df de los anexos

    # Carga de archivos principales

    for file in sorted(ruta.glob("EPA_*")):
        if file.suffix == ".tab":
            df = pd.read_csv(file, sep="\t", encoding="utf-8", low_memory=False)
        else:
            df = pd.read_csv(file, sep="\t", encoding="utf-8", low_memory=False)
            # esto lo dejamos así porque, aunque la extensión sea distinta,
            # en la práctica también van separados por tabulación

        key = file.stem
        dfs[key] = df

    # Carga de anexos

    for file in sorted(ruta.glob("EPAAnexo_*")):
        if file.suffix == ".tab":
            df = pd.read_csv(file, sep="\t", encoding="utf-8", low_memory=False)
        else:
            df = pd.read_csv(file, encoding="utf-8", low_memory=False)

        key = file.stem
        dfs_a[key] = df

    return dfs, dfs_a

In [3]:
dfs, dfs_a = cargar_csv()

In [4]:
dfs["EPA_2021T1"]

,CICLO,CCAA,PROV,NVIVI,NIVEL,NPERS,EDAD1,RELPP1,SEXO1,NCONY,...,SIDI3,SIDAC1,SIDAC2,DAUSVAC,DAUSENF,DAUSOTR,TRAANT,AOI,RELLB4,FACTOREL
0,194,16,1,1,1,1,40,1,1,2,...,,1,1,2.0,NaN,NaN,1,04,,193.90
1,194,16,1,1,1,2,30,2,6,1,...,,1,1,NaN,NaN,NaN,1,03,,193.90
2,194,16,1,2,1,1,35,1,6,2,...,,1,1,1.0,NaN,NaN,1,04,,252.32
3,194,16,1,2,1,2,30,2,1,1,...,,1,1,1.0,NaN,NaN,1,04,,252.32
4,194,16,1,2,2,3,0,3,6,0,...,,,,NaN,NaN,NaN,,,,252.32
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
142845,194,52,52,57996,1,2,25,3,6,0,...,,1,1,NaN,NaN,NaN,6,04,,155.14
142846,194,52,52,57996,1,3,20,3,6,0,...,,2,2,NaN,NaN,NaN,6,05,,155.14
142847,194,52,52,57997,1,1,30,1,1,2,...,,1,1,NaN,NaN,NaN,1,04,,439.75
142848,194,52,52,57997,1,2,30,2,6,1,...,,1,1,NaN,NaN,NaN,1,04,,439.75


In [ ]:
# Crear la carpeta de salida si no existe
ruta_salida = Path("../data/interim")
ruta_salida.mkdir(parents=True, exist_ok=True)

for nombre, df in dfs.items():
    # Hacemos una copia para no tocar el original
    df_nuevo = df.rename(columns={"HORPLU": "HOREPLU"})
    
    # Guardamos el nuevo CSV
    df_nuevo.to_csv(ruta_salida / f"{nombre}.csv", index=False)

A continuación vamos a hacer otra función para cargar los nuevos csv que son los que vamos a mergear.

In [2]:
def cargar_csv_int():
    """
    Carga los archivos principales de la EPA y los anexos
    desde data/interim y devuelve dos diccionarios:
    - dfs: archivos principales
    - dfs_a: anexos
    """

    ruta = Path("../data/interim")  # carpeta donde están los datos

    dfs = {}    # aquí guardamos los df principales
    dfs_a = {}  # aquí guardamos los df de los anexos

    # Carga de archivos principales

    for file in sorted(ruta.glob("EPA_*")):
        if file.suffix == ".tab":
            df = pd.read_csv(file, sep="\t", encoding="utf-8", low_memory=False)
        else:
            df = pd.read_csv(file, sep="\t", encoding="utf-8", low_memory=False)
            # esto lo dejamos así porque, aunque la extensión sea distinta,
            # en la práctica también van separados por tabulación

        key = file.stem
        dfs[key] = df

    # Carga de anexos

    for file in sorted(ruta.glob("EPAAnexo_*")):
        if file.suffix == ".tab":
            df = pd.read_csv(file, sep="\t", encoding="utf-8", low_memory=False)
        else:
            df = pd.read_csv(file, encoding="utf-8", low_memory=False)

        key = file.stem
        dfs_a[key] = df

    return dfs, dfs_a

In [3]:
dfs, dfs_a = cargar_csv_int()

Aquí comprobamos, de nuevo, que todas las columnas se llaman igual

In [4]:
lista_columnas = []
for e in dfs.values():
    c= set(e.columns)
    lista_columnas.append(c)

i = 0
ref = lista_columnas[0]

for c in lista_columnas:
    i +=1
    if c == lista_columnas[0]:
        print(f"El grupo {i} es igual, todo bien.")
    else:
        faltan = ref - c
        sobran = c - ref
        print(f"En el grupo{i} faltan estas columnas: \n{faltan}\n y sobran estas columnas \n{sobran}\n")
              

El grupo 1 es igual, todo bien.
El grupo 2 es igual, todo bien.
El grupo 3 es igual, todo bien.
El grupo 4 es igual, todo bien.
El grupo 5 es igual, todo bien.
El grupo 6 es igual, todo bien.
El grupo 7 es igual, todo bien.
El grupo 8 es igual, todo bien.
El grupo 9 es igual, todo bien.
El grupo 10 es igual, todo bien.
El grupo 11 es igual, todo bien.
El grupo 12 es igual, todo bien.
El grupo 13 es igual, todo bien.
El grupo 14 es igual, todo bien.
El grupo 15 es igual, todo bien.
El grupo 16 es igual, todo bien.
El grupo 17 es igual, todo bien.
El grupo 18 es igual, todo bien.
El grupo 19 es igual, todo bien.
El grupo 20 es igual, todo bien.


In [ ]:
ruta_entrada = Path("../data/interim")
ruta_salida = Path("../data/interim")
ruta_salida.mkdir(parents=True, exist_ok=True)

archivos = [
    f for f in glob.glob(os.path.join(ruta_entrada, "*.csv"))
    if Path(f).name != "EPA_joined.csv"
]

# Leer con coma, porque así están ahora los archivos intermedios
dfs = [pd.read_csv(f) for f in archivos]

# Concatenar
df_final = pd.concat(dfs, ignore_index=True)

# Guardar tabulado
archivo_final = ruta_salida / "EPA_joined.csv"
df_final.to_csv(archivo_final, sep="\t", index=False)


In [29]:
# Comprobación
df_epa_joined = pd.read_csv(archivo_final, sep="\t")

print("Archivo guardado en:", archivo_final)
print("Dimensiones finales:", df_epa_joined.shape)
print("Número de columnas:", len(df_epa_joined.columns))
print("¿Está HOREPLU?", "HOREPLU" in df_epa_joined.columns)
print("¿Sigue HORPLU?", "HORPLU" in df_epa_joined.columns)

/var/folders/18/2jngcg0x5csd3tyb3xj47rrm0000gn/T/ipykernel_58729/1329759060.py:2: DtypeWarning: Columns (12,13,14,15,17,20,22,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,75,76,77,78,79,80,81,82,83,87,88,89) have mixed types. Specify dtype option on import or set low_memory=False.
  df_epa_joined = pd.read_csv(archivo_final, sep="\t")


Archivo guardado en: ../data/interim/EPA_joined.csv
Dimensiones finales: (2516028, 91)
Número de columnas: 91
¿Está HOREPLU? True
¿Sigue HORPLU? False


In [30]:
df_epa_joined.head()

,CICLO,CCAA,PROV,NVIVI,NIVEL,NPERS,EDAD1,RELPP1,SEXO1,NCONY,...,SIDI3,SIDAC1,SIDAC2,DAUSVAC,DAUSENF,DAUSOTR,TRAANT,AOI,RELLB4,FACTOREL
0,198,16,1,1,1,1,50,1,6,2,...,,1,1,2.0,NaN,NaN,1,04,,209.58
1,198,16,1,1,1,2,50,2,1,1,...,,1,1,2.0,NaN,NaN,1,04,,209.58
2,198,16,1,1,1,3,16,3,6,0,...,,,,NaN,NaN,NaN,6,09,,209.58
3,198,16,1,1,2,4,10,3,1,0,...,,,,NaN,NaN,NaN,,,,209.58
4,198,16,1,2,1,1,40,1,1,2,...,,1,1,4.0,NaN,NaN,1,04,,113.69


In [31]:
df_epa_joined.sample(10)

,CICLO,CCAA,PROV,NVIVI,NIVEL,NPERS,EDAD1,RELPP1,SEXO1,NCONY,...,SIDI3,SIDAC1,SIDAC2,DAUSVAC,DAUSENF,DAUSOTR,TRAANT,AOI,RELLB4,FACTOREL
2401916,210,7,5,3153,1,3,25,3,1,0,...,NaN,NaN,NaN,NaN,NaN,NaN,6.0,9.0,NaN,145.66
2437582,210,16,20,18076,1,2,55,2,6,1,...,NaN,1.0,1.0,1.0,NaN,NaN,1.0,4.0,NaN,403.71
2342143,197,15,31,32607,1,2,40,2,6,1,...,,1,1,1.0,NaN,NaN,,04,,239.45
373348,209,16,48,47222,1,2,65,2,1,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9.0,NaN,485.85
1599013,195,7,34,37303,2,5,0,3,1,0,...,,,,NaN,NaN,NaN,,,,121.12
2391060,197,2,50,52932,1,2,65,2,6,1,...,,,,NaN,NaN,NaN,,09,,165.03
1863512,196,7,34,35757,1,3,20,3,1,0,...,,1,1,NaN,NaN,NaN,,04,,275.66
2145594,211,10,3,1829,1,1,50,1,6,0,...,NaN,1.0,1.0,NaN,NaN,NaN,NaN,4.0,NaN,602.69
1695544,202,2,22,20716,2,3,5,3,1,0,...,,,,NaN,NaN,NaN,,,,104.64
2456519,210,13,28,26020,1,2,35,2,6,1,...,NaN,1.0,1.0,6.0,NaN,NaN,1.0,4.0,NaN,1565.63


In [32]:
df_epa_joined.tail()

,CICLO,CCAA,PROV,NVIVI,NIVEL,NPERS,EDAD1,RELPP1,SEXO1,NCONY,...,SIDI3,SIDAC1,SIDAC2,DAUSVAC,DAUSENF,DAUSOTR,TRAANT,AOI,RELLB4,FACTOREL
2516023,210,52,52,50903,1,3,16,3,6,0,...,NaN,NaN,NaN,NaN,NaN,NaN,6.0,9.0,NaN,93.23
2516024,210,52,52,50903,2,4,10,3,6,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,93.23
2516025,210,52,52,50903,2,5,10,3,6,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,93.23
2516026,210,52,52,50904,1,1,60,1,6,0,...,NaN,1.0,1.0,NaN,NaN,NaN,1.0,4.0,NaN,23.97
2516027,210,52,52,50904,1,2,50,7,6,0,...,NaN,NaN,NaN,NaN,NaN,NaN,6.0,9.0,NaN,23.97


In [34]:
len(df_epa_joined["CICLO"].unique())

20

In [38]:
display(df_epa_joined.describe)

<bound method NDFrame.describe of          CICLO  CCAA  PROV  NVIVI  NIVEL  NPERS  EDAD1  RELPP1  SEXO1  NCONY  \
0          198    16     1      1      1      1     50       1      6      2   
1          198    16     1      1      1      2     50       2      1      1   
2          198    16     1      1      1      3     16       3      6      0   
3          198    16     1      1      2      4     10       3      1      0   
4          198    16     1      2      1      1     40       1      1      2   
...        ...   ...   ...    ...    ...    ...    ...     ...    ...    ...   
2516023    210    52    52  50903      1      3     16       3      6      0   
2516024    210    52    52  50903      2      4     10       3      6      0   
2516025    210    52    52  50903      2      5     10       3      6      0   
2516026    210    52    52  50904      1      1     60       1      6      0   
2516027    210    52    52  50904      1      2     50       7      6      0   

     

In [46]:
df_epa_joined["RELLB1"].notnull

<bound method Series.notnull of 0             
1             
2             
3             
4             
          ... 
2516023    NaN
2516024    NaN
2516025    NaN
2516026    NaN
2516027    NaN
Name: RELLB1, Length: 2516028, dtype: object>